# Scraping Vienna's Property Market

## Scraping Multiple Pages of Data

In [44]:
import os
import re
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
from datetime import datetime

# Define Chrome WebDriver path dynamically based on the current working directory
current_directory = os.getcwd()
chrome_driver_path = os.path.join(current_directory, 'chromedriver')
service = Service(chrome_driver_path)
options = webdriver.ChromeOptions()

# Disable images to speed up loading
prefs = {"profile.managed_default_content_settings.images": 2}
options.add_experimental_option("prefs", prefs)

# Run in headless mode for faster scraping
options.add_argument("--headless")
options.add_argument("--log-level=3")  # Suppress most logs
options.page_load_strategy = "eager"

driver = webdriver.Chrome(service=service, options=options)

# Set timeouts
driver.set_script_timeout(500)  # Script execution timeout
driver.set_page_load_timeout(500)  # Page load timeout

# Base URL and data list
base_url = "https://www.willhaben.at/iad/immobilien/eigentumswohnung/wien?page="
data = []
progress_file = "progress.txt"
temp_file = "scraped_properties_temp.csv"

# Save scraped data to a CSV file
def save_scraped_data(data, filename=temp_file):
    if data:
        df = pd.DataFrame(data)
        df.to_csv(filename, index=False)
        print(f"Data saved to {filename}")

# Read the last progress from the file
def read_progress():
    if os.path.exists(progress_file):
        with open(progress_file, "r") as file:
            return int(file.read().strip())
    return 1  # Start from page 1 if no progress file exists

# Update progress to a file
def update_progress(page):
    with open(progress_file, "w") as file:
        file.write(str(page))

try:
    # Start scraping from where it left off
    start_page = read_progress()
    print(f"Resuming from page {start_page}...")

    for page in range(start_page, 101):  # Scrape up to page 100
        print(f"Scraping page {page}...")
        try:
            driver.get(base_url + str(page))

            # Scroll through the page to load all content
            scroll_pause_time = 0.5  # Pause between scrolls
            total_scrolls = 30  # Number of scrolls based on page length

            for i in range(total_scrolls):
                driver.execute_script(f"window.scrollTo(0, {i * 500});")
                time.sleep(scroll_pause_time)

            # Wait for elements to load
            WebDriverWait(driver, 30).until(
                EC.presence_of_all_elements_located((By.CLASS_NAME, "hPOcQO"))
            )

            # Collect data from the page
            elements_hPOcQO = driver.find_elements(By.CLASS_NAME, "hPOcQO")
            for element in elements_hPOcQO:
                text = element.text.strip()
                url = element.get_attribute("href")
                if text:
                    data.append({"text": text, "url": url})
            update_progress(page)  # Save progress after successful scrape

        except TimeoutException:
            print(f"Timeout on page {page}. Retrying...")
            continue  # Retry the same page
        except Exception as e:
            print(f"Error on page {page}: {e}")
            save_scraped_data(data)  # Save data before exiting
            break  # Stop the loop for debugging

finally:
    driver.quit()
    save_scraped_data(data)  # Save whatever was scraped

# Extract specific fields from the scraped data
def extract_field(entry):
    text = entry['text']
    title = text.split('\n')[0]  # Title is the first line
    postcode_match = re.search(r'\b\d{4}\b', text)  # Look for a 4-digit postcode
    postcode = postcode_match.group(0) if postcode_match else None
    size_match = re.search(r'\d+\s*m²', text)  # Look for size in m²
    size = size_match.group(0) if size_match else None
    price_match = re.search(r'€\s[\d.,]+', text)  # Look for price
    price = price_match.group(0) if price_match else None
    return {
        "Title": title,
        "Postcode": postcode,
        "Size": size,
        "Price": price,
        "URL": entry['url']
    }

# Process the scraped data with regex
regex_data = [extract_field(entry) for entry in data]

# Create a DataFrame for the cleaned data
df_regex = pd.DataFrame(regex_data)

# Generate a descriptive filename
current_date = datetime.now().strftime("%Y-%m-%d")
first_page = read_progress()
last_page = first_page + len(data) // len(df_regex.columns) - 1  # Estimate the last page
file_name = f"scraped_properties_pages_{first_page}_to_{last_page}_{current_date}.csv"

# Save the final data to the file
df_regex.to_csv(file_name, index=False)
print(f"Scraping complete. Data saved to '{file_name}'.")


Resuming from page 27...
Scraping page 27...
Scraping page 28...
Scraping page 29...
Scraping page 30...
Scraping page 31...
Scraping page 32...
Scraping page 33...
Scraping page 34...
Scraping page 35...
Scraping page 36...
Scraping page 37...
Scraping page 38...
Scraping page 39...
Scraping page 40...
Scraping page 41...
Scraping page 42...
Scraping page 43...
Scraping page 44...
Scraping page 45...
Scraping page 46...
Scraping page 47...
Scraping page 48...
Scraping page 49...
Scraping page 50...
Scraping page 51...
Scraping page 52...
Scraping page 53...
Scraping page 54...
Scraping page 55...
Scraping page 56...
Scraping page 57...
Scraping page 58...
Scraping page 59...
Scraping page 60...
Scraping page 61...
Scraping page 62...
Scraping page 63...
Scraping page 64...
Scraping page 65...
Scraping page 66...
Scraping page 67...
Scraping page 68...
Scraping page 69...
Scraping page 70...
Scraping page 71...
Scraping page 72...
Scraping page 73...
Scraping page 74...
Scraping page 7

## Data Cleaning: Preparing the Dataset

In [195]:
import os
import pandas as pd

# Define the file path dynamically based on the current working directory
current_directory = os.getcwd()
file_path = os.path.join(current_directory, 'scraped_properties.csv')

# Detect delimiter and load the dataset
with open(file_path, 'r') as file:
    first_line = file.readline()
    delimiter = ',' if ',' in first_line else (';' if ';' in first_line else '\t')

# Load the dataset
data = pd.read_csv(file_path, delimiter=delimiter, on_bad_lines='skip')

# Clean the data
data = data.dropna(subset=['Size', 'Price', 'Postcode'])
data['Size'] = data['Size'].str.replace(r'\D+', '', regex=True).astype(int)

# Modify the Price column to remove numbers after the comma
data['Price'] = data['Price'].str.replace(r',(\d+)$', '', regex=True)

# Remove all non-digit characters for numerical processing
data['Price'] = data['Price'].str.replace(r'\D+', '', regex=True).astype(int)

# Remove outliers based on the 2nd and 98th percentiles
size_lower_bound = data['Size'].quantile(0.02)
size_upper_bound = data['Size'].quantile(0.98)
price_lower_bound = data['Price'].quantile(0.02)
price_upper_bound = data['Price'].quantile(0.98)

data = data[
    (data['Size'] >= size_lower_bound) & 
    (data['Size'] <= size_upper_bound) & 
    (data['Price'] >= price_lower_bound) & 
    (data['Price'] <= price_upper_bound)
]

# Calculate Price/m2
data['Price/m2'] = data['Price'] / data['Size']

# Save the cleaned data dynamically to the same directory
cleaned_file_path = os.path.join(current_directory, 'cleaned_properties_with_price_per_m2.csv')
data.to_csv(cleaned_file_path, index=False)

# Display a preview of the cleaned data
print("Cleaned Data Preview:")
print(data.head())


Cleaned Data Preview:
                                               Title  Postcode  Size   Price  \
0                                  EINMALIGE CHANCE!      1040    66  319000   
1  Luxuriöser Jugendstil-Erstbezug: 89 m² mit 3 Z...      1030    89  779000   
2                              ETWAS GANZ BESONDERES      1020    63  379000   
3  Charmante 2-Zimmer Stilaltbauwohnung im Herzen...      1130    50  329000   
4  Großzügige, renovierungsbedürftige 3-Zimmer-Wo...      1120    71  199000   

                                                 URL     Price/m2  
0  https://www.willhaben.at/iad/immobilien/d/eige...  4833.333333  
1  https://www.willhaben.at/iad/immobilien/d/eige...  8752.808989  
2  https://www.willhaben.at/iad/immobilien/d/eige...  6015.873016  
3  https://www.willhaben.at/iad/immobilien/d/eige...  6580.000000  
4  https://www.willhaben.at/iad/immobilien/d/eige...  2802.816901  


# Data Visualisation

In [196]:
import pandas as pd
import plotly.express as px
import json
from ipywidgets import Dropdown, interact

# Load datasets
data = pd.read_csv('cleaned_properties_with_price_per_m2.csv')
with open('vienna.geojson', 'r') as f:
    vienna_geojson = json.load(f)

# Map district names to postal codes
district_to_postcode = {
    "Innere Stadt": "1010", "Leopoldstadt": "1020", "Landstraße": "1030",
    "Wieden": "1040", "Margareten": "1050", "Mariahilf": "1060",
    "Neubau": "1070", "Josefstadt": "1080", "Alsergrund": "1090",
    "Favoriten": "1100", "Simmering": "1110", "Meidling": "1120",
    "Hietzing": "1130", "Penzing": "1140", "Rudolfsheim-Fünfhaus": "1150",
    "Ottakring": "1160", "Hernals": "1170", "Währing": "1180",
    "Döbling": "1190", "Brigittenau": "1200", "Floridsdorf": "1210",
    "Donaustadt": "1220", "Liesing": "1230"
}

# Add properties to GeoJSON
for feature in vienna_geojson['features']:
    district_name = feature['properties']['name']
    feature['properties']['Postcode'] = district_to_postcode.get(district_name)
    feature['properties']['District'] = district_name

# Aggregate property data
stats = data.groupby('Postcode').agg(
    Median_Price_m2=('Price/m2', 'median'),
    Mean_Price_m2=('Price/m2', 'mean'),
    Observations=('Price/m2', 'count')
).reset_index()

# Prepare data for merging
stats['Postcode'] = stats['Postcode'].astype(str)
geojson_data = pd.DataFrame([
    {'Postcode': feature['properties']['Postcode'], 'District': feature['properties']['District']}
    for feature in vienna_geojson['features']
])
geojson_data['Postcode'] = geojson_data['Postcode'].astype(str)

# Merge stats with GeoJSON data
stats = stats.merge(geojson_data, on='Postcode', how='left')

# Round prices and add formatted columns for hover data
stats[['Median_Price_m2', 'Mean_Price_m2']] = stats[['Median_Price_m2', 'Mean_Price_m2']].round(0).astype(int)
stats['Median Price (price/m²)'] = stats['Median_Price_m2'].apply(lambda x: f"{x} €")
stats['Mean Price (price/m²)'] = stats['Mean_Price_m2'].apply(lambda x: f"{x} €")

# Create choropleth map
def create_map(color_column):
    max_val = 16000
    min_val = stats[color_column].min()

    hover_data = {
        'District': True,
        'Observations': True,
        'Median Price (price/m²)': True if color_column == 'Median_Price_m2' else False,
        'Mean Price (price/m²)': True if color_column == 'Mean_Price_m2' else False,
        color_column: False  # Exclude the raw column used for coloring
    }

    fig = px.choropleth(
        stats, geojson=vienna_geojson, locations='Postcode',
        featureidkey='properties.Postcode', color=color_column,
        color_continuous_scale="Blues", range_color=[min_val, max_val],
        title=f"Vienna Property Prices ({'Mean' if color_column == 'Mean_Price_m2' else 'Median'}) per m²",
        hover_data=hover_data
    )
    fig.update_geos(fitbounds="locations", visible=False)
    fig.update_layout(
        coloraxis_colorbar=dict(title=None),
        margin={"r": 0, "t": 50, "l": 0, "b": 90},
        annotations=[dict(
            x=0.5, y=-0.15, showarrow=False, text="This map shows property prices per m² in Vienna. Hover over districts for details.",
            xref="paper", yref="paper", align="center", font=dict(size=12)
        )]
    )
    fig.show()

# Interactive dropdown for map
def interactive_map(view):
    create_map(view)

interact(interactive_map, view=Dropdown(
    options=[('Median Price/m²', 'Median_Price_m2'), ('Mean Price/m²', 'Mean_Price_m2')],
    value='Median_Price_m2', description='Map View:', style={'description_width': 'initial'}
))


interactive(children=(Dropdown(description='Map View:', options=(('Median Price/m²', 'Median_Price_m2'), ('Mea…

<function __main__.interactive_map(view)>

In [197]:
import plotly.express as px
from ipywidgets import interact, Dropdown

# Function to create a bar chart for the top 5 districts
def create_bar_chart(column):
    top_districts = stats.nlargest(5, column)
    top_districts = top_districts[top_districts['District'].notna()]

    # If fewer than 5 rows, fill in additional districts
    if len(top_districts) < 5:
        remaining_districts = stats[~stats['District'].isin(top_districts['District']) & stats['District'].notna()]
        additional_districts = remaining_districts.nlargest(5 - len(top_districts), column)
        top_districts = pd.concat([top_districts, additional_districts])

    # Debug: Print the top 5 districts to verify selection
    print(f"Top 5 districts for {column}:\n", top_districts)

    # Create the bar chart
    fig = px.bar(
        top_districts,
        x='District',
        y=column,
        title=f"Top 5 Districts by {'Mean' if column == 'Mean_Price_m2' else 'Median'} Price per m² (in €)",
        labels={'District': 'District', column: 'Price per m² (€)'}
    )

    # Customize the layout
    fig.update_layout(
        xaxis_title='District',
        yaxis_title='Price per m² (€)',
        yaxis=dict(range=[0, 20000]),  # Adjust scale if necessary
        margin={"r": 0, "t": 50, "l": 50, "b": 50},
        plot_bgcolor='white',  
        paper_bgcolor='white', 
        showlegend=False
    )

    # Add euro/m² labels on the bars at the bottom
    fig.update_traces(
        text=[f"€{int(price):,}" for price in top_districts[column]],
        textposition='inside',  
        textfont=dict(
            size=12,
            color='white', 
            family='Arial',
            weight='bold' 
        )
    )

    fig.show()

# Interactive dropdown for bar chart
def interactive_bar(view):
    create_bar_chart(view)

dropdown_bar = Dropdown(
    options=[
        ('Median Price/m²', 'Median_Price_m2'),
        ('Mean Price/m²', 'Mean_Price_m2')
    ],
    value='Median_Price_m2',
    description='Bar Chart View:',
    style={'description_width': 'initial'}
)

interact(interactive_bar, view=dropdown_bar)


interactive(children=(Dropdown(description='Bar Chart View:', options=(('Median Price/m²', 'Median_Price_m2'),…

<function __main__.interactive_bar(view)>

In [198]:
import pandas as pd
import plotly.express as px
from sklearn.preprocessing import PolynomialFeatures
import numpy as np

# Load the dataset
file_path = 'cleaned_properties_with_price_per_m2.csv'
data = pd.read_csv(file_path)

# Step 1: Prepare the data
# Drop rows with missing values and limit Size to 400 m²
data = data.dropna(subset=['Size', 'Price/m2'])
data = data[data['Size'] <= 400]

# Step 2: Define clusters based on Price/m²
def classify_price(price):
    if price > 10000:  # Threshold for luxury
        return 'Luxury'
    elif price < 5000:  # Threshold for affordable
        return 'Affordable'
    else:
        return 'Mid-range'

data['Cluster'] = data['Price/m2'].apply(classify_price)

# Step 3: Scatter plot with polynomial regression lines
fig = px.scatter(
    data,
    x='Size',
    y='Price/m2',
    color='Cluster',
    title="Property Prices by Size and Cluster",
    labels={'Size': 'Size (m²)', 'Price/m2': 'Price per m² (€)', 'Cluster': 'Category'},
    hover_data=['Postcode']
)

# Step 4: Add polynomial regression lines for each cluster
# Fit polynomial regression models for each cluster
clusters = data['Cluster'].unique()
regression_lines = []

for cluster in clusters:
    cluster_data = data[data['Cluster'] == cluster]
    X = cluster_data[['Size']].values
    y = cluster_data['Price/m2'].values

    if len(cluster_data) > 1:  # Fit regression if enough data points exist
        poly = PolynomialFeatures(degree=2, include_bias=False)
        X_poly = poly.fit_transform(X)

        model = LinearRegression()
        model.fit(X_poly, y)
        regression_lines.append({
            'Cluster': cluster,
            'Coefficients': model.coef_,
            'Intercept': model.intercept_,
            'Poly': poly
        })

# Add polynomial regression lines to the scatter plot
x_range = np.linspace(0, 250, 300).reshape(-1, 1)
for line in regression_lines:
    y_range = line['Poly'].transform(x_range) @ line['Coefficients'] + line['Intercept']
    fig.add_scatter(
        x=x_range.flatten(),
        y=y_range,
        mode='lines',
        name=f"{line['Cluster']} Trendline",
        line=dict(dash='dash')
    )

# Customize layout for better visibility
fig.update_layout(
    plot_bgcolor='white',
    xaxis=dict(
        gridcolor='lightgray',
        title="Property Size (m²)"
    ),
    yaxis=dict(
        gridcolor='lightgray',
        title="Price per m² (€)"
    ),
    legend=dict(
        title="Cluster",
        bordercolor="black",
        borderwidth=1
    ),
    title=dict(
        x=0.5,  # Center the title
        font=dict(size=20)
    ),
    margin={"r": 0, "t": 50, "l": 50, "b": 50}
)

fig.show()
